# 06. Agentic RAG — 도구 2개 조합 + Langfuse Tracing


> **시나리오 안내** — 이 실습은 가상의 이커머스 마켓플레이스 **쇼핑몰**(자체 PB 라인: 베이직 · 프리미엄 · 라이트 · 한정판)를 소재로 합니다.

| 항목 | 내용 |
|------|------|
| 목적 | LLM이 **도구를 스스로 골라서 조합**해 답하는 Agentic RAG 패턴을 구현 |
| LLM | OpenAI `gpt-4o-mini` (function calling 지원) |
| Embedding | OpenAI `text-embedding-3-small` (1536차원) |
| 벡터 DB | **ChromaDB** (Colab 로컬 디스크) |
| 모니터링 | Langfuse Cloud (**03/04/05와 동일 Project** 재사용) |
| 데이터 소스 | 01 실습의 `product-cs-sft-lab/documents/` (5개 카테고리, ~50개 문서) |
| 런타임 | Colab **CPU 런타임** (GPU 불필요 — Embedding/LLM 모두 OpenAI API) |
| 예상 소요 | ~30~40분 |
| 예상 비용 | ~$0.05 (Embedding ~$0.01 + Agent 호출 ~$0.04) |

> **사전 조건**
> - 01 실습 결과물 `product-cs-sft-lab/documents/` 폴더 (5개 카테고리 하위에 .txt 파일들)
> - OpenAI API Key (gpt-4o-mini + Embedding 호출)
> - 03/04에서 만든 Langfuse Project 키 그대로

---

## 왜 Agentic RAG인가?

- **6강 RAG**: "사용자 질문을 임베딩 → 벡터 DB에서 검색 → 결과를 LLM에 넣어 답변" — **검색이 항상 1번**.
- **7강 한계**: "구조화된 데이터(메타데이터)는 벡터 검색이 비효율" — 카테고리·날짜 같은 정확 매칭은 SQL/JSON 필터가 더 적합.
- **8강 Agentic Search**: "LLM에게 도구 2개를 주고, 질문에 따라 **알아서 골라 쓰게** 한다." → **Agentic RAG**.

이번 실습은 14·15강 SFT 라인과는 **별개의 트랙**입니다. SFT는 모델의 **행동을 데이터로 박는** 접근이었고, RAG는 **모델의 지식을 외부 검색으로 보완**하는 접근이에요. 둘은 보완재라 실무에서는 같이 씁니다.

## 비교 프레임워크 (3 Level 시나리오)

같은 합성 문서 50여 개를 두 도구(`vector_search`, `metadata_filter`)로 노출시킨 뒤, **세 난이도의 질문**을 던져 Agent의 도구 선택 패턴을 관찰합니다.

- **Level 1** — 의미 기반 질문 → Agent가 `vector_search` **1회** 선택
- **Level 2** — 카테고리/제품 필터 질문 → Agent가 `metadata_filter` **1회** 선택
- **Level 3** — 복합 질문 → Agent가 `metadata_filter` → `vector_search` **2회 조합**

각 Level의 결과는 Langfuse Trace로 자동 기록되고, 대시보드에서 **호출 체인 트리**로 시각화됩니다.


---
## Phase 1: 환경 설정 — 패키지 설치

이 실습은 **GPU가 필요 없습니다.** Colab 런타임을 **CPU로** 두고 시작하세요.

설치 대상:

- **`openai`** — gpt-4o-mini 호출 (function calling) + Embedding API
- **`chromadb`** — Colab 로컬 디스크에 벡터 인덱스를 저장하는 임베디드 벡터 DB
- **`langfuse`** — 03/04/05와 동일 Project에 Trace 기록
- **`tiktoken`** — OpenAI tokenizer (chunk 길이 계산용)


In [ ]:
# 패키지 설치
#
# ⚠️ OpenTelemetry 버전 정렬:
#   chromadb는 opentelemetry-exporter-otlp-proto-grpc를 의존성으로 끌어오는데,
#   Colab에 미리 설치된 opentelemetry-sdk와 버전이 어긋나면
#   "ImportError: cannot import name 'OTEL_PYTHON_SDK_INTERNAL_METRICS_ENABLED'"
#   가 발생합니다 (sdk가 exporter보다 구버전).
#   → opentelemetry 패키지 전부를 동시 upgrade해서 버전을 강제 정렬합니다.
!pip install --quiet -U openai chromadb langfuse tiktoken tqdm "gradio>=5.0,<6" \
    opentelemetry-api opentelemetry-sdk \
    opentelemetry-exporter-otlp-proto-common \
    opentelemetry-exporter-otlp-proto-grpc \
    opentelemetry-proto

# Colab은 일부 opentelemetry 모듈을 import 시점에 캐시합니다. 새로 설치한 버전이 적용되도록
# 이미 로드된 opentelemetry.* 모듈을 정리합니다 (런타임 재시작 없이 적용).
import sys
for mod in list(sys.modules):
    if mod.startswith("opentelemetry"):
        del sys.modules[mod]

import openai
import chromadb
import langfuse as _lf_mod
import gradio as gr

print(f"✅ openai {openai.__version__}")
print(f"✅ chromadb {chromadb.__version__}")
print(f"✅ langfuse {_lf_mod.__version__}")
print(f"✅ gradio {gr.__version__}")

---
## Phase 2: 인증 — OpenAI / Langfuse

03/04에서 쓴 **동일한 Langfuse Project 키**를 그대로 입력합니다. 같은 Project에 Trace를 쌓아서 한 대시보드에서 "**SFT 평가 → 양자화 → Gradio 데모 → Agentic RAG**"까지 시계열로 봅니다.

- **`OPENAI_API_KEY`** — gpt-4o-mini + Embedding 호출
- **`LANGFUSE_PUBLIC_KEY` / `SECRET_KEY`** — 03 Project 것 그대로
- **`LANGFUSE_BASE_URL`** — `https://jp.cloud.langfuse.com` (Japan 리전, 한국에서 지연 최적)


In [ ]:
import os
from getpass import getpass

# (1) OpenAI
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ").strip()

# (2) Langfuse — 03/04와 동일 Project 재사용
if not os.environ.get("LANGFUSE_PUBLIC_KEY"):
    os.environ["LANGFUSE_PUBLIC_KEY"] = getpass("LANGFUSE_PUBLIC_KEY (pk-lf-...): ").strip()
if not os.environ.get("LANGFUSE_SECRET_KEY"):
    os.environ["LANGFUSE_SECRET_KEY"] = getpass("LANGFUSE_SECRET_KEY (sk-lf-...): ").strip()
os.environ.setdefault("LANGFUSE_BASE_URL", "https://jp.cloud.langfuse.com")
os.environ["LANGFUSE_HOST"] = os.environ["LANGFUSE_BASE_URL"]

print("\n✅ 인증 완료")
print(f"   Langfuse host: {os.environ['LANGFUSE_BASE_URL']}")


---
## Phase 3: 합성 문서 로드 + 메타데이터 자동 추출

01 실습에서 만든 `product-cs-sft-lab/documents/` 폴더를 스캔합니다. 다섯 개 카테고리(`troubleshooting`, `dev-guides`, `qa-procedures`, `customer-support`, `meeting-notes`) 하위에 `.txt` 파일들이 있어요.

이번 실습은 **두 도구가 같은 문서 풀을 다른 방식으로 다루는** 게 핵심입니다.

- **벡터 검색용 컨텐츠**: 파일 내용 전문 (다음 셀에서 임베딩)
- **메타데이터 필터용 인덱스**: 파일명·카테고리·문자수에서 추출한 구조화 정보

01의 결과물이 로컬 또는 Drive에 있으면 자동 감지합니다.


In [ ]:
from pathlib import Path
import re

LOCAL_DOCS = Path("product-cs-sft-lab/documents")
DRIVE_DOCS = Path("/content/drive/MyDrive/product-cs-sft-lab/documents")

if LOCAL_DOCS.exists():
    DOCS_DIR = LOCAL_DOCS
    print(f"📂 로컬 경로 사용: {DOCS_DIR}")
elif DRIVE_DOCS.exists():
    DOCS_DIR = DRIVE_DOCS
    print(f"☁️  Drive 경로 사용: {DOCS_DIR}")
else:
    from google.colab import drive
    drive.mount("/content/drive")
    DOCS_DIR = DRIVE_DOCS
    assert DOCS_DIR.exists(), f"01 실습 결과물이 없습니다: {DOCS_DIR}"

# 문서 + 메타데이터 추출
PRODUCT_KEYWORDS = [
    "무선 청소기", "노트북", "무선 청소기",
    "Foldable Phone", "Flip Phone6", "Fold", "Flip", "접이식",
    "Smartwatch7", "Smartwatch", "워치",
    "Wireless Earbuds Pro", "디지털 액세서리", "버즈",
    "보안 솔루션", "쇼핑몰 Health", "쇼핑몰헬스",
]

def extract_products(text: str) -> list:
    found = []
    for kw in PRODUCT_KEYWORDS:
        if kw.lower() in text.lower() and kw not in found:
            found.append(kw)
    return found

documents = []  # [{doc_id, category, filename, title, content, products, char_count}, ...]

for cat_dir in sorted(DOCS_DIR.iterdir()):
    if not cat_dir.is_dir():
        continue
    category = cat_dir.name
    for txt in sorted(cat_dir.glob("*.txt")):
        content = txt.read_text(encoding="utf-8").strip()
        if len(content) < 50:
            continue
        title = re.sub(r"^\d+_", "", txt.stem).replace("_", " ")
        documents.append({
            "doc_id": f"{category}/{txt.stem}",
            "category": category,
            "filename": txt.name,
            "title": title,
            "content": content,
            "products": extract_products(content + " " + title),
            "char_count": len(content),
        })

# 카테고리별 분포 출력
from collections import Counter
cat_counts = Counter(d["category"] for d in documents)
print(f"\n✅ 문서 {len(documents)}개 로드 완료\n")
print(f"{'카테고리':<22}{'문서 수':>8}")
print("-" * 32)
for cat, n in sorted(cat_counts.items()):
    print(f"{cat:<22}{n:>8}")

print(f"\n샘플 첫 문서:")
print(f"  doc_id:    {documents[0]['doc_id']}")
print(f"  title:     {documents[0]['title']}")
print(f"  products:  {documents[0]['products']}")
print(f"  char_count: {documents[0]['char_count']}")


---
## Phase 4: Chunking + Embedding + ChromaDB 인덱싱

5강(Chunking) + 6강(Embedding)을 코드로 직접 실행하는 단계입니다.

**파이프라인 3단계**:
1. **Chunking** — 긴 문서를 ~400 토큰 단위로 자릅니다 (Embedding 입력 한도 + 검색 정확도 균형).
2. **Embedding** — `text-embedding-3-small`(1536차원)로 각 청크를 벡터화. OpenAI API 1회 호출에 ~100개씩 묶어서 보냅니다.
3. **ChromaDB 저장** — `persist_directory="./chroma_db"`에 영구 저장. 다시 실행하면 재사용 가능.

> 💡 **메타데이터도 같이 저장합니다.** ChromaDB는 `where={...}` 필터로 검색 범위를 좁힐 수 있는데, 이 실습에서는 두 도구가 분리돼 있어서 ChromaDB 메타데이터는 결과 표시용으로만 씁니다 (실제 필터링은 다음 셀의 `metadata_filter` 도구가 별도로 담당).


In [ ]:
import tiktoken
from openai import OpenAI
from tqdm import tqdm

oai = OpenAI()
EMBED_MODEL = "text-embedding-3-small"
EMBED_DIM = 1536
CHUNK_TOKENS = 400
CHUNK_OVERLAP = 50

enc = tiktoken.get_encoding("cl100k_base")


def chunk_by_tokens(text: str, size: int = CHUNK_TOKENS, overlap: int = CHUNK_OVERLAP) -> list[str]:
    tokens = enc.encode(text)
    chunks = []
    i = 0
    while i < len(tokens):
        chunk = enc.decode(tokens[i:i + size])
        chunks.append(chunk)
        if i + size >= len(tokens):
            break
        i += size - overlap
    return chunks


# --- 1) Chunking ---
all_chunks = []  # [{chunk_id, doc_id, category, products, text}]
for doc in documents:
    pieces = chunk_by_tokens(doc["content"])
    for j, piece in enumerate(pieces):
        all_chunks.append({
            "chunk_id": f"{doc['doc_id']}#{j}",
            "doc_id": doc["doc_id"],
            "category": doc["category"],
            "products": ",".join(doc["products"]) or "none",
            "text": piece,
        })
print(f"✅ 청크 {len(all_chunks)}개 생성 (문서 {len(documents)}개에서)")


# --- 2) Embedding ---
def embed_batch(texts: list[str], batch_size: int = 100) -> list[list[float]]:
    out = []
    for i in tqdm(range(0, len(texts), batch_size), desc="임베딩 생성"):
        batch = texts[i:i + batch_size]
        resp = oai.embeddings.create(model=EMBED_MODEL, input=batch)
        out.extend([d.embedding for d in resp.data])
    return out


embeddings = embed_batch([c["text"] for c in all_chunks])
print(f"✅ 임베딩 {len(embeddings)}개 생성 (차원 {len(embeddings[0])})")


# --- 3) ChromaDB ---
import chromadb
chroma_client = chromadb.PersistentClient(path="./chroma_db")
COLLECTION_NAME = "product-cs-docs"

# 기존 컬렉션 있으면 삭제 후 재생성 (재실행 안전)
try:
    chroma_client.delete_collection(COLLECTION_NAME)
except Exception:
    pass
collection = chroma_client.create_collection(name=COLLECTION_NAME, metadata={"hnsw:space": "cosine"})

collection.add(
    ids=[c["chunk_id"] for c in all_chunks],
    embeddings=embeddings,
    documents=[c["text"] for c in all_chunks],
    metadatas=[{
        "doc_id": c["doc_id"],
        "category": c["category"],
        "products": c["products"],
    } for c in all_chunks],
)
print(f"\n✅ ChromaDB 인덱싱 완료: {collection.count()}개 청크 저장")


---
## Phase 5: 도구 정의 — `vector_search` + `metadata_filter`

7강의 핵심 메시지를 두 함수로 구현합니다.

- **`vector_search(query, top_k)`** — 비구조화 데이터를 **의미**로 검색.
  - 입력 질문을 임베딩 → ChromaDB cosine 유사도 → 상위 청크 반환.
  - **언제 쓰는가**: "운동화 사이즈 교환 절차 해결법" 같은 의미 기반 질문.

- **`metadata_filter(category, product, sort_by, top_k)`** — 구조화 데이터를 **정확 매칭**으로 필터링.
  - 카테고리/제품 키워드/문자수로 documents 리스트를 필터.
  - **언제 쓰는가**: "무선 청소기 관련 트러블슈팅 문서 목록" 같은 메타데이터 기반 질문.

각 함수에는 **OpenAI function calling용 JSON schema**도 함께 정의합니다. Agent가 이 schema를 보고 "어떤 인자로 도구를 부를지" 결정합니다.


In [ ]:
# --- 도구 1: vector_search ---
def vector_search(query: str, top_k: int = 5) -> list[dict]:
    """의미 기반 벡터 검색. 청크 단위로 반환."""
    qvec = oai.embeddings.create(model=EMBED_MODEL, input=[query]).data[0].embedding
    res = collection.query(query_embeddings=[qvec], n_results=top_k)
    out = []
    for cid, doc, meta, dist in zip(res["ids"][0], res["documents"][0], res["metadatas"][0], res["distances"][0]):
        out.append({
            "chunk_id": cid,
            "doc_id": meta["doc_id"],
            "category": meta["category"],
            "score": round(1 - dist, 4),
            "preview": doc[:200] + ("…" if len(doc) > 200 else ""),
        })
    return out


# --- 도구 2: metadata_filter ---
def metadata_filter(
    category: str = "",
    product: str = "",
    sort_by: str = "char_count",
    top_k: int = 5,
) -> list[dict]:
    """카테고리/제품으로 문서 목록을 필터링. 정확 매칭."""
    matched = []
    for d in documents:
        if category and d["category"] != category:
            continue
        if product:
            if not any(product.lower() in p.lower() for p in d["products"]):
                continue
        matched.append(d)
    if sort_by == "char_count":
        matched.sort(key=lambda x: x["char_count"], reverse=True)
    return [{
        "doc_id": d["doc_id"],
        "title": d["title"],
        "category": d["category"],
        "products": d["products"],
        "char_count": d["char_count"],
    } for d in matched[:top_k]]


# --- OpenAI function calling용 schema ---
TOOL_SPECS = [
    {
        "type": "function",
        "function": {
            "name": "vector_search",
            "description": (
                "비구조화 자연어 질의를 의미 기반으로 검색합니다. "
                "구체적인 증상·해결법·기술 설명처럼 단어 매칭이 어려운 질문에 사용하세요."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "검색할 자연어 질의"},
                    "top_k": {"type": "integer", "default": 5},
                },
                "required": ["query"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "metadata_filter",
            "description": (
                "카테고리·제품명 같은 메타데이터로 문서 목록을 필터링합니다. "
                "'무슨 문서가 있는가', '특정 제품의 트러블슈팅 목록' 같은 구조화 질문에 사용하세요."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "category": {
                        "type": "string",
                        "enum": ["troubleshooting", "dev-guides", "qa-procedures", "customer-support", "meeting-notes", ""],
                        "default": "",
                        "description": "문서 카테고리. 빈 문자열이면 전체.",
                    },
                    "product": {
                        "type": "string",
                        "default": "",
                        "description": "제품명 키워드 (예: '무선 청소기', '접이식', 'Watch'). 빈 문자열이면 전체.",
                    },
                    "sort_by": {"type": "string", "enum": ["char_count"], "default": "char_count"},
                    "top_k": {"type": "integer", "default": 5},
                },
            },
        },
    },
]

TOOL_FUNCS = {"vector_search": vector_search, "metadata_filter": metadata_filter}

# --- 새너티 체크 ---
print("[vector_search 샘플]")
for r in vector_search("주방 가전 디스플레이 깜빡임", top_k=2):
    print(f"  · {r['doc_id']} (score={r['score']}) — {r['preview'][:80]}…")

print("\n[metadata_filter 샘플 — category=troubleshooting]")
for r in metadata_filter(category="troubleshooting", top_k=3):
    print(f"  · {r['doc_id']} ({r['char_count']}자) — {r['title']}")


---
## Phase 6: Agent 구성 + Langfuse Tracing

8강의 **"LLM에게 도구를 주고 알아서 골라 쓰게 한다"**를 OpenAI function calling으로 직접 구현합니다.

**Agent 동작 루프**:
1. 사용자 질문 + 도구 schema → `gpt-4o-mini`에 전달
2. 모델이 `tool_calls`를 반환 → 우리가 해당 함수 실행 → 결과를 `tool` role로 추가
3. 도구 결과를 다시 모델에 입력 → 모델이 또 도구 호출 또는 최종 답변 결정
4. 최종 답변(`finish_reason="stop"`)이 나올 때까지 반복

**Langfuse Tracing 구조** (이번 실습의 핵심):
- 한 질문 = **Trace 하나**
- Agent의 각 LLM 호출 = **`generation` observation**
- 각 도구 실행 = **`span` observation**
- 대시보드에서 **호출 체인 트리**로 시각화 → "Agent가 어떤 도구를 어떤 순서로 골랐는지" 한눈에.


In [ ]:
import json
import time
from langfuse import get_client, propagate_attributes

langfuse = get_client()
print(f"✅ Langfuse 클라이언트 초기화 (SDK v{_lf_mod.__version__})")

AGENT_MODEL = "gpt-4o-mini"
MAX_AGENT_TURNS = 5  # tool 호출 무한루프 방지

AGENT_SYSTEM_PROMPT = (
    "당신은 이커머스 고객 지원 도메인 사내 문서 에이전트입니다. "
    "사용자 질문에 답하려면 제공된 도구를 적절히 골라 사용하세요. "
    "단순 의미 검색이면 vector_search, 카테고리·제품 필터링이면 metadata_filter를 쓰고, "
    "복합 질문은 두 도구를 순차적으로 조합하세요. "
    "도구 결과를 종합해 한국어로 간결하게 답변하세요."
)


def run_agent(user_question: str, level_label: str = "agentic-rag") -> dict:
    """질문 한 건을 Agent 루프로 처리. Langfuse Trace 자동 기록.
    
    핵심: `start_as_current_observation` 컨텍스트 매니저로 부모-자식 자동 연결.
    이렇게 해야 대시보드에서 호출 체인 트리가 들여쓰기로 그려짐.
    """
    messages = [
        {"role": "system", "content": AGENT_SYSTEM_PROMPT},
        {"role": "user", "content": user_question},
    ]
    tool_history = []

    with propagate_attributes(tags=["agentic-rag", level_label]):
        # --- Trace 루트: span (자식들의 부모) ---
        with langfuse.start_as_current_observation(
            as_type="span",
            name="agentic-rag-turn",
            input={"question": user_question},
            metadata={"level": level_label, "model": AGENT_MODEL},
        ) as root_span:
            trace_id = root_span.trace_id
            final_answer = ""

            for turn in range(MAX_AGENT_TURNS):
                # --- LLM 호출 (generation observation, 루트의 자식) ---
                t0 = time.perf_counter()
                resp = oai.chat.completions.create(
                    model=AGENT_MODEL,
                    messages=messages,
                    tools=TOOL_SPECS,
                    temperature=0,
                )
                latency_ms = int((time.perf_counter() - t0) * 1000)
                msg = resp.choices[0].message
                usage = resp.usage

                with langfuse.start_as_current_observation(
                    as_type="generation",
                    name=f"agent-turn-{turn}",
                    model=AGENT_MODEL,
                    input=messages.copy(),
                    metadata={"latency_ms": latency_ms},
                ) as gen:
                    gen.update(
                        output=msg.content or (msg.tool_calls[0].function.name if msg.tool_calls else ""),
                        usage_details={
                            "input": usage.prompt_tokens,
                            "output": usage.completion_tokens,
                            "total": usage.total_tokens,
                        },
                    )

                # --- 종료 조건: tool_calls 없으면 최종 답변 ---
                if not msg.tool_calls:
                    final_answer = msg.content or ""
                    break

                # --- 도구 호출 분기 ---
                messages.append({
                    "role": "assistant",
                    "content": msg.content,
                    "tool_calls": [{
                        "id": tc.id,
                        "type": "function",
                        "function": {"name": tc.function.name, "arguments": tc.function.arguments},
                    } for tc in msg.tool_calls],
                })

                for tc in msg.tool_calls:
                    fname = tc.function.name
                    fargs = json.loads(tc.function.arguments or "{}")
                    print(f"  🔧 도구 선택: {fname}({fargs})")

                    with langfuse.start_as_current_observation(
                        as_type="span",
                        name=f"tool:{fname}",
                        input=fargs,
                    ) as tool_span:
                        result = TOOL_FUNCS[fname](**fargs)
                        tool_span.update(output={"result_count": len(result), "preview": result[:3]})

                    tool_history.append({"name": fname, "args": fargs, "result_count": len(result)})
                    messages.append({
                        "role": "tool",
                        "tool_call_id": tc.id,
                        "content": json.dumps(result, ensure_ascii=False),
                    })
            else:
                final_answer = "(turn limit exceeded)"

            root_span.update(output={"answer": final_answer, "tool_history": tool_history})

    return {
        "answer": final_answer,
        "tool_history": tool_history,
        "trace_id": trace_id,
    }


print("\n✅ run_agent 정의 완료 (start_as_current_observation 컨텍스트 패턴)")


---
## Phase 7: 단계별 질문 — Level 1 → 2 → 3

세 난이도의 질문을 차례로 던져서 **Agent의 도구 선택 패턴**을 직접 관찰합니다.

| Level | 질문 유형 | 기대 동작 |
|-------|----------|---------|
| 1 | 의미 기반 트러블슈팅 | `vector_search` 1회 |
| 2 | 카테고리·제품 필터 | `metadata_filter` 1회 |
| 3 | 복합 질문 | `metadata_filter` → `vector_search` (조합) |

각 Level의 호출 직후 **Langfuse Trace ID**가 출력됩니다. 이 ID로 대시보드에서 호출 체인 트리를 즉시 찾아볼 수 있어요.


In [ ]:
LEVEL_QUESTIONS = [
    ("level-1-vector-only",
     "무선 청소기 카메라 강력 모드에서 색상 보정가 이상하게 잡히는 문제를 어떻게 해결하나요?"),
    ("level-2-metadata-only",
     "트러블슈팅 카테고리에 어떤 문서들이 있는지 목록을 보여주세요."),
    ("level-3-combined",
     "주방 가전 관련 문서 중에서 가장 자주 보고되는 하드웨어 이슈와 그 대응 방안을 정리해 주세요."),
]

results = []
for level, q in LEVEL_QUESTIONS:
    print("=" * 75)
    print(f"[{level}]")
    print(f"질문: {q}")
    print("-" * 75)
    res = run_agent(q, level_label=level)
    print(f"\n💬 답변:\n{res['answer']}")
    print(f"\n🪜 도구 호출 순서: {[t['name'] for t in res['tool_history']]}")
    print(f"🔗 Trace ID: {res['trace_id']}")
    results.append({"level": level, "question": q, **res})
    print()

langfuse.flush()
print("✅ 모든 Level 완료 + Langfuse Trace 기록")


---
## Phase 8: Langfuse 대시보드 확인 — 호출 체인 트리 (클라이맥스)

방금 실행한 세 Trace가 03/04/05와 같은 Project에 누적됐습니다. 대시보드에서 **Level 1 vs Level 3의 트리 모양 차이**를 비교.

### 확인 순서

1. 대시보드 → Traces 탭 → Filter `tag = 'agentic-rag'`
2. **Level 1 Trace** 클릭 → 트리: `agentic-rag-turn → tool:vector_search → agent-turn-0/1`
3. **Level 3 Trace** 클릭 → 트리: `agentic-rag-turn → tool:metadata_filter → tool:vector_search → agent-turn-0/1/2`
4. 각 Generation 노드에서 **프롬프트/응답 전문 + 토큰 수 + latency**까지 모두 보임
5. 같은 Project의 다른 태그(`sft-eval`, `quantization`, `gradio-demo`)와 함께 **시계열로 한 화면에 누적**되는 것 확인

### 비용·시간 정리

- Level 1: gpt-4o-mini 호출 2회(도구 결정 + 답변 생성) + Embedding 1회
- Level 3: gpt-4o-mini 호출 3회 + Embedding 1회 + tool 실행 2회
- Level 3 Trace의 총 비용·총 시간이 한눈에 보입니다 — **"도구 조합이 비싸지만 답변 품질이 더 정확하다"**의 트레이드오프 실측.


In [ ]:
print("=" * 70)
print("🎯 확인 체크리스트")
print("=" * 70)

print(f"\n[1] Langfuse 대시보드")
print(f"    {os.environ['LANGFUSE_BASE_URL']}")
print(f"    → Traces 탭 → Filter: tag = 'agentic-rag'")

print(f"\n[2] Level별 Trace ID (방금 생성됨)")
for r in results:
    print(f"    [{r['level']}] {r['trace_id']}")

print(f"\n[3] 같은 Project에서 04 Phase까지 누적")
print(f"    - 03 tag: 'sft-eval'        (4조건 평가)")
print(f"    - 04 tag: 'quantization'    (양자화 품질)")
print(f"    - 05 tag: 'gradio-demo'     (실 사용자 대화)")
print(f"    - 06 tag: 'agentic-rag'     (도구 조합)  ← 지금 추가됨")
print(f"    → 학습 → 평가 → 양자화 → 실 사용 → Agentic RAG 까지 한 대시보드")

print(f"\n[4] 도구 호출 패턴 요약")
for r in results:
    seq = " → ".join(t["name"] for t in r["tool_history"]) or "(직접 답변)"
    print(f"    [{r['level']}] {seq}")


---
## Phase 9: Gradio UI — Agent 채팅 + 도구 호출 실시간 모니터링

지금까지는 `LEVEL_QUESTIONS` 세 개로 미리 짠 시나리오만 돌렸어요. 이제 **누구나 질문을 자유롭게 던지고 Agent가 어떤 도구를 어떤 순서로 부르는지 눈으로 볼 수 있는 UI**를 만듭니다.

**핵심 설계**

- **Gradio `ChatInterface`** + `type="messages"` (OpenAI 호환 포맷, 05에서 본 것과 같음)
- **`gr.ChatMessage(metadata={"title": ...})`** — Gradio 5.x의 공식 도구 호출 표시 패턴. **접을 수 있는 박스**로 도구 이름·인자·결과가 표시되고, 그 아래 최종 답변이 보통 메시지로 들어갑니다.
- **`yield` 스트리밍** — 도구 호출이 진행되는 매 단계마다 UI가 즉시 갱신. "🔧 호출 중..." → "✅ 완료 (N건)"으로 상태가 바뀌는 게 실시간으로 보임
- **콘솔 로그도 같이** — `print`로 stdout에도 같은 정보를 찍어서 강의 시연 시 모니터링 두 곳(UI + Colab 셀 출력) 동시에 볼 수 있음
- **Langfuse Trace 자동 기록** — 기존 `run_agent`와 같은 `start_observation` 패턴. 대시보드에는 `gradio-agent-chat` 태그로 누적

**사용법**

1. 셀 실행 → Gradio UI가 셀 출력 영역(또는 share URL)에 뜸
2. 예시 질문 클릭하거나 직접 질문 입력
3. **Agent가 도구를 부를 때마다 채팅창에 접을 수 있는 회색 박스**가 추가됨 — 클릭하면 인자·결과 보임
4. 마지막에 **최종 답변**이 일반 메시지로 표시
5. 대시보드(Langfuse)에서 같은 대화의 Trace를 호출 체인 트리로 다시 확인


In [ ]:
import gradio as gr
import json as _json

# 추가 태그 — Level 1~3 시나리오와 구분
GRADIO_TAGS = ["agentic-rag", "gradio-agent-chat"]


def chat_fn_agentic(message: str, history: list):
    """Gradio ChatInterface가 호출할 함수.
    Agent의 도구 호출을 ChatMessage로 스트리밍하면서 콘솔에도 같은 로그를 print."""
    messages_for_ui = []  # UI에 yield할 ChatMessage 리스트

    # --- LLM 입력 메시지 조립 ---
    messages = [{"role": "system", "content": AGENT_SYSTEM_PROMPT}]
    for h in history:
        # Gradio messages 포맷: {"role": "user"|"assistant", "content": "..."}
        if isinstance(h, dict) and h.get("role") in ("user", "assistant"):
            content = h.get("content", "")
            # ChatMessage(metadata=...) 도구 박스는 history에서 제외 (역할 모호)
            if isinstance(content, str) and content:
                messages.append({"role": h["role"], "content": content})
    messages.append({"role": "user", "content": message})

    print("━" * 70)
    print(f"[chat] user: {message!r}")

    with propagate_attributes(tags=GRADIO_TAGS):
        # ⚠️ Generator(yield) 함수에서는 context manager가 OTel 컨텍스트를 깨므로
        # 명시적 start_observation + parent 객체 메서드로 부모-자식 연결.
        root_span = langfuse.start_observation(
            as_type="span",
            name="gradio-agent-chat",
            input={"question": message},
            metadata={"deployment": "colab_gradio"},
        )
        try:
            for turn in range(MAX_AGENT_TURNS):
                # --- LLM 호출 ---
                resp = oai.chat.completions.create(
                    model=AGENT_MODEL,
                    messages=messages,
                    tools=TOOL_SPECS,
                    temperature=0,
                )
                msg = resp.choices[0].message

                # Generation observation — root_span 객체에서 직접 자식 생성
                gen = root_span.start_observation(
                    as_type="generation",
                    name=f"agent-turn-{turn}",
                    model=AGENT_MODEL,
                    input=messages.copy(),
                )
                gen.update(output=msg.content or "(tool_call)")
                gen.end()

                # --- 종료 조건: 최종 답변 ---
                if not msg.tool_calls:
                    final = msg.content or ""
                    print(f"[chat] final answer ({len(final)} chars)")
                    messages_for_ui.append(
                        gr.ChatMessage(role="assistant", content=final)
                    )
                    yield messages_for_ui
                    root_span.update(output={"answer": final, "turns": turn + 1})
                    return

                # --- 도구 호출 분기 ---
                messages.append({
                    "role": "assistant",
                    "content": msg.content,
                    "tool_calls": [{
                        "id": tc.id,
                        "type": "function",
                        "function": {"name": tc.function.name, "arguments": tc.function.arguments},
                    } for tc in msg.tool_calls],
                })

                for tc in msg.tool_calls:
                    fname = tc.function.name
                    fargs = _json.loads(tc.function.arguments or "{}")
                    args_str = _json.dumps(fargs, ensure_ascii=False)

                    print(f"  🔧 {fname}({args_str})")

                    # (1) 도구 "호출 중" 메시지 추가 → UI 즉시 갱신
                    messages_for_ui.append(gr.ChatMessage(
                        role="assistant",
                        content=f"**인자**\n```json\n{args_str}\n```",
                        metadata={"title": f"🔧 {fname} 호출 중..."},
                    ))
                    yield messages_for_ui

                    # (2) 실제 함수 실행 + Langfuse span — root_span의 자식으로 명시
                    tool_span = root_span.start_observation(
                        as_type="span",
                        name=f"tool:{fname}",
                        input=fargs,
                    )
                    result = TOOL_FUNCS[fname](**fargs)
                    tool_span.update(output={"result_count": len(result), "preview": result[:3]})
                    tool_span.end()

                    print(f"     → {len(result)}건 반환")

                    # (3) 결과로 같은 메시지 업데이트 → UI 다시 갱신
                    preview_str = _json.dumps(result[:3], ensure_ascii=False, indent=2)
                    if len(preview_str) > 800:
                        preview_str = preview_str[:800] + "\n... (truncated)"
                    messages_for_ui[-1] = gr.ChatMessage(
                        role="assistant",
                        content=(
                            f"**인자**\n```json\n{args_str}\n```\n\n"
                            f"**결과 {len(result)}건 (앞 3개 미리보기)**\n```json\n{preview_str}\n```"
                        ),
                        metadata={"title": f"✅ {fname} 완료 ({len(result)}건)"},
                    )
                    yield messages_for_ui

                    messages.append({
                        "role": "tool",
                        "tool_call_id": tc.id,
                        "content": _json.dumps(result, ensure_ascii=False),
                    })

            # MAX_TURNS 초과
            messages_for_ui.append(gr.ChatMessage(
                role="assistant",
                content="(turn limit exceeded — Agent가 결론을 못 냈습니다)",
            ))
            yield messages_for_ui
        finally:
            root_span.end()
            langfuse.flush()


# --- ChatInterface 정의 ---
demo_agent = gr.ChatInterface(
    fn=chat_fn_agentic,
    type="messages",
    title="🤖 쇼핑몰 고객 지원팀 Agentic RAG",
    description=(
        "질문하면 Agent가 **`vector_search`(의미 검색)**와 **`metadata_filter`(메타데이터 필터)** 중 "
        "필요한 도구를 골라 부릅니다. 도구 호출 박스를 클릭하면 인자·결과를 펼쳐볼 수 있어요. "
        "모든 대화는 03~05와 같은 Langfuse Project에 `gradio-agent-chat` 태그로 자동 기록됩니다."
    ),
    examples=[
        ["무선 청소기 주문한 운동화 사이즈 교환 절차 방법은?"],
        ["트러블슈팅 카테고리에 어떤 문서들이 있는지 목록을 보여주세요"],
        ["주방 가전 관련 하드웨어 이슈와 대응 방안을 정리해주세요"],
        ["보안 솔루션 보안 관련 dev-guide 문서 중 가장 자세한 것은?"],
        ["Smartwatch 알림 설정 방법 알려주세요"],
    ],
    theme=gr.themes.Soft(),
)

# Colab 환경에서 share URL 자동 생성 시도. 실패하면 inline 안내.
# Gradio 버전 체크 (5.x 미만이면 type='messages' 미지원 → 명확한 안내)
import gradio as _gr
_major = int(_gr.__version__.split(".")[0])
if _major < 5:
    print(f"❌ Gradio {_gr.__version__} 감지 — 이 셀은 5.x가 필요합니다.")
    print("   해결: 셀 2(Phase 1 설치 셀)를 다시 실행해서 gradio를 5.x로 업그레이드하고 런타임을 재시작하세요.")
else:
    try:
        demo_agent.launch(share=True, inline=True, debug=False)
    except Exception as e:
        print(f"⚠️  launch 실패: {e}")
        print("    → Colab+Gradio 5.x 조합에서 가끔 발생. HF Spaces 배포는 05 패턴 참고.")


---
## 완료 — 5개 노트북 전체 정리

| 단계 | 결과물 | 핵심 메시지 |
|---|---|---|
| 01 데이터 생성 | `train.jsonl` / `test.jsonl` + `documents/` | 합성 데이터 부트스트랩 |
| 02 LoRA SFT | FP16 merged 모델 | 모델에 행동을 박는 학습 |
| 03 4조건 평가 | Langfuse Trace + Judge Score | SFT 고유 기여도 정량화 |
| 04 GGUF 양자화 | FP16 / Q8_0 / Q4_K_M GGUF | 크기 vs 품질 트레이드오프 |
| 05 Gradio + Spaces | 공개 데모 URL + 실 사용 Trace | 비개발자도 쓸 수 있는 배포 |
| **06 Agentic RAG** | **도구 2개 조합 + 호출 체인 Trace** | **모델 외부 지식·도구 활용** |

**오늘 달성한 것**

1. ✅ **두 도구를 LLM에 노출** — `vector_search`(의미) + `metadata_filter`(구조)
2. ✅ **Agent가 자율적으로 도구 선택** — Level 1/2/3 시나리오로 패턴 관찰
3. ✅ **호출 체인을 Langfuse Trace로 시각화** — 13강 "블랙박스를 여는 도구" 체험
4. ✅ **MLOps 전체 루프 완성** — 한 Project에 학습/평가/양자화/배포/RAG 누적

**SFT vs RAG — 둘은 보완재**

- **SFT**(02~05) — 모델의 **행동·톤**을 데이터로 박음. 시스템 프롬프트만으로 못 바꾸는 어미·인사말 같은 형식 행동에 강함.
- **RAG**(06) — 모델의 **지식·문서**를 외부에서 끌어옴. 학습 시점에 없던 문서·최신 정보 대응에 강함.
- 실무 결합 예: "SFT 모델 + RAG 도구" → 톤은 일관되게, 사실은 최신 문서로.